In [1]:
import math
import numpy as np
import pandas as pd
import scipy.stats as st
import scipy.optimize as opt
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.io import imread, imsave

np.random.seed(52)


In [2]:
eps = 0.05
k = 10
n = 100
grid = np.arange(k)
freq = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
unit = np.ones(k)


In [3]:
def kolmogorov_approx(x):
    res = 1.0
    for j in range(1, 1001):
        res += 2 * (-1) ** j * np.exp(-2 * j * j * x * x)
    return res


def norm_target(params):
    a, b = params
    vals = st.norm.cdf(grid[1:], loc=a, scale=b)

    mid = 1.0
    for i in range(k - 2):
        mid *= (vals[i + 1] - vals[i]) ** freq[i + 1]

    left = vals[0] ** freq[0]
    right = (1 - vals[-1]) ** freq[-1]
    return -(left * mid * right)



In [4]:
exp_u = 10 * unit
chi_u = float(((freq - exp_u) ** 2 / exp_u).sum())
p_u_chi = float(1 - st.chi2(k - 1).cdf(chi_u))

print('Равномерная модель R(0,10), критерий хи-квадрат')
print(f'chi^2 = {chi_u:.6f}; p-value = {p_u_chi:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_u_chi > eps else 'отвергается')

f_emp = np.array([freq[:i].sum() for i in range(len(freq) + 1)]) / n
f_uni = np.array([unit[:i].sum() for i in range(len(unit))]) / 10

d_u = float(
    np.sqrt(n) * max(
        max(abs(f_emp[i] - f_uni[i]), abs(f_emp[i + 1] - f_uni[i]))
        for i in range(k)
    )
)
p_u_ks = 1 - kolmogorov_approx(d_u)

print()
print('Равномерная модель R(0,10), критерий Колмогорова')
print(f'Δ = {d_u:.2f}; p-value = {p_u_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_u_ks > eps else 'отвергается')


fit = opt.differential_evolution(
    func=norm_target,
    bounds=[(0, 10), (0, 10)],
    maxiter=10000
)

m_hat = float(fit.x[0])
s_hat = float(fit.x[1])
g = st.norm.cdf(grid[1:], loc=m_hat, scale=s_hat)

exp_n = [n * g[0]]
for i in range(k - 2):
    exp_n.append(n * (g[i + 1] - g[i]))
exp_n.append(n * (1 - g[-1]))
exp_n = np.array(exp_n)

chi_n = float(((freq - exp_n) ** 2 / exp_n).sum())
p_n_chi = float(1 - st.chi2(k - 3).cdf(chi_n))

print()
print('Нормальная модель N(theta1, theta2^2), критерий хи-квадрат')
print(f'theta1 = {m_hat:.6f}; theta2 = {s_hat:.6f}')
print(f'chi^2 = {chi_n:.6f}; p-value = {p_n_chi:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_n_chi > eps else 'отвергается')


B = 50000
sample = []
for i in range(k):
    sample.extend([i] * freq[i])
sample = np.array(sample)

m0 = sample.mean()
s0 = sample.std(ddof=1)


def norm_cdf(x, a, b):
    return 0.5 * (1 + math.erf((x - a) / (np.sqrt(2) * b)))


d_n = max(
    np.sqrt(n) * max(
        abs(norm_cdf(grid[i], m0, s0) - f_emp[i]),
        abs(norm_cdf(grid[i], m0, s0) - f_emp[i + 1])
    )
    for i in range(k)
)


def boot(rep):
    ans = []
    model = st.norm(loc=m0, scale=s0)

    for _ in range(rep):
        y = np.sort(model.rvs(size=n))
        e = [i / n for i in range(n + 1)]
        my = y.mean()
        sy = y.std(ddof=1)

        stat = max(
            np.sqrt(n) * max(
                abs(norm_cdf(y[t], my, sy) - e[t]),
                abs(norm_cdf(y[t], my, sy) - e[t + 1])
            )
            for t in range(len(y))
        )
        ans.append(stat)

    return ans


print()
print('Нормальная модель N(theta1, theta2^2), критерий Колмогорова')
print(f'Δ = {d_n:.6f}')

boot_vals = boot(B)
p_n_ks = np.mean(np.array(boot_vals) >= d_n)

print(f'p-value: {p_n_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_n_ks > eps else 'отвергается')

Равномерная модель R(0,10), критерий хи-квадрат
chi^2 = 16.400000; p-value = 0.0590
Решение по H0: нет оснований для отклонения гипотезы

Равномерная модель R(0,10), критерий Колмогорова
Δ = 1.40; p-value = 0.0397
Решение по H0: отвергается

Нормальная модель N(theta1, theta2^2), критерий хи-квадрат
theta1 = 5.292873; theta2 = 2.675816
chi^2 = 9.799736; p-value = 0.2002
Решение по H0: нет оснований для отклонения гипотезы

Нормальная модель N(theta1, theta2^2), критерий Колмогорова
Δ = 1.002094
p-value: 0.0153
Решение по H0: отвергается
